Use the preprocessed data found in "preprocessed_data.csv" which was created by the other Jupyter notebook. We start with creating a correlation heatmap of the variables

In [5]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

NORMALIZATION_METHOD = "number of player's possessions"

if NORMALIZATION_METHOD is None:
    df = pd.read_csv('preprocessed_data.csv')
else:
    df = pd.read_csv(f'preprocessed_normalized_{NORMALIZATION_METHOD}.csv')

print(df.columns.tolist())

['points', "points per player's possession", 'field goals made', 'field goals attempted', 'field goals, %', '3-pt field goals made', '3-pt field goals attempted', '3-pt field goals, %', 'free throws made', 'free throws attempted', 'free throws, %', 'rebounds', 'offensive rebounds', 'defensive rebounds', 'assists', 'steals', 'turnovers', 'blocks', 'fouls', 'fouls drawn', 'plus/minus', '2-pt field goals made', '2-pt field goals attempted', '2-pt field goals, %', 'points off assists', 'screen assist', 'points off screen assists', "number of player's possessions", 'team points with player', 'offensive rating', 'opponent possessions played', "opponent's points with player", 'defensive rating', 'net rating', 'assists to turnovers', 'steals to turnovers', 'draw foul rate', 'true shooting percentage', 'effective field goal percentage', 'uncontested field goals made', 'uncontested field goals', 'uncontested field goals, %', 'contested field goals made', 'contested field goals', 'contested field

First, plot a heatmap of the correlation matrix.

In [ ]:
correlation_matrix = df.corr()
plt.figure(figsize=(40, 35))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm")
plt.show()

Next, we create a histogram for each of the 123 features. This should give us an insight about the distributions as well as if some have too many zero/NaN values. If that that is the case, we should consider dropping these features. Always check the heatmap if dropping them is feasible.

In [7]:
features = df.columns

# Split the data into 6 groups
feature_sets = [df.columns[i:i+20] for i in range(0, len(df.columns), 20)]

# For each set of features, create a 4 by 5 grid of hisgroams
def plot_and_save_histograms(feature_sets):
    for idx, features in enumerate(feature_sets):
        # Melt the DataFrame for the current set of features
        df_subset = pd.melt(df[features], var_name='Feature', value_name='Value')
        
        # Create a FacetGrid, plotting a 4x5 grid of histograms
        g = sns.FacetGrid(df_subset, col='Feature', col_wrap=5, sharex=False, sharey=False, height=2.5)
        # Plot histograms for each feature
        g.map(plt.hist, 'Value', bins=50, color='m', edgecolor='black')
        # Plot estimated density for each feature
        # g.map(sns.kdeplot, 'Value', color='m', fill=True, alpha=0.5)
        
        # Customizing and saving each plot
        plt.subplots_adjust(top=0.9)
        if NORMALIZATION_METHOD is None:
            g.fig.suptitle(f'Feature Set {idx+1}')
        else:
            g.fig.suptitle(f'Feature Set {idx+1} (Normalized by {NORMALIZATION_METHOD} with power-transformed values)')
        g.set_titles('{col_name}')
        g.set_axis_labels('Value', 'Frequency')
        g.fig.tight_layout(rect=[0, 0, 1, 0.95])  # Adjust the rect to fit the suptitle
        
        # Save the figure if needed
        plt.savefig(f'histogram_set_{idx+1}.png')
        plt.close()
        # plt.show()

plot_and_save_histograms(feature_sets)


/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498:

#### Discussion of what histograms showed:

* Many approximately normally distributed variables and many which seem to follow an exponential (or similar) distribution.
* Quite a few features seem to have missing values. This can be seen in the histograms which follow a distribution but for the max/min values there is a spike in the plot
* We can try different data imputation methods and visually inspect that these do not significantly alter the distributions.
* Generally, the "percentage" features always seem kind of redundant because they depend on the total attempts and attempts made anyways. Might just remove them.

Due to the reasons stated above, the following features have been removed (percentage and "many" insignificant number of attempts). These are specialized shooting/scoring statistics and thus have smaller sample size:
* "(un)contested fields goald, %", "opponent's field goals, %", "transition attacks, %", 
* "catch and shoot shots made, %", "catch and drive shots made, %", "screens off, %", "post up, %", "isolations, %", "hand off, %", 
* "cuts, %", "pr handler, %", "pr roller, %", "pick-n-pops, %", "opponent transition shots made, %", "opp catch and shoot shots made, %", 
* "opp catch and drive shots made, %", "opp screens off shots made, %", "opponent post up shots made, %", 
* "opponent isolation shots made, %", "opponent hand off shots made, %", "opponent cuts shots made, %", "opponent pick-n-roll shots made, %", 
* "opponent pick-n-pop shots made, %", "drives, %", "right drives made, %", "left drives made, %", "opponent drives shots made, %"

The above histograms are stored in "./histograms" and the file "histogram_set_4.png" shows that we have very limited data for pick-n-roll (roller). Hence, we just remove them altogether. These additionally include:
* "pnr rollers made", "pnr rollers attempted"
* For now we still include the "handlers" as there is a little bit more data available for them.

After removing these features it seems like data imputation might not be necessary. While still many features have predominantly many (close to) zero values, this might be very reasonable to assume. E.g. many players do not attempt any 3-pt shots. These would then correspond to zero-inflated features and we might have to deal with them separately. 

In [6]:
REMOVED_FEATURES = ["contested field goals, %", "uncontested field goals, %", "opponent's field goals, %", "transition attacks, %", 
                  "catch and shoot shots made, %", "catch and drive shots made, %", "screens off, %", "post up, %", "isolation, %", "hand off, %", 
                  "cuts, %", "pr handler, %", "pr roller, %", "pick-n-pops, %", "opponent transition shots made, %", "opp catch and shoot shots made, %",
                  "opp catch and drive shots made, %", "opponent screens off shots made, %", "opponent post up shots made, %",
                  "opponent isolation shots made, %", "opponent hand off shots made, %", "opponent cuts shots made, %", "opponent pick-n-roll shots made, %",
                  "opponent pick-n-pop shots made, %", "drives, %", "right drives made, %", "left drives made, %", "opponent drives shots made, %",
                  "pnr rollers made", "pnr rollers attempted"]

df = df.drop(REMOVED_FEATURES, axis=1)
# Count how many features still have "%" in their name and list them
print(df.columns[df.columns.str.contains('%')].tolist())
print(f"Number of features left: {len(df.columns)}")
post_feature_sets = [df.columns[i:i+20] for i in range(0, len(df.columns), 20)]
plot_and_save_histograms(post_feature_sets)

# Save the preprocessed data
df.to_csv('train_data.csv', index=False)
print(df.columns.tolist())

['field goals, %', '3-pt field goals, %', 'free throws, %', '2-pt field goals, %']
Number of features left: 93


/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498: FutureWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, CategoricalDtype) instead
  if pd.api.types.is_categorical_dtype(vector):
/opt/homebrew/Caskroom/miniforge/base/envs/iml-project/lib/python3.12/site-packages/seaborn/_oldcore.py:1498:

['points', "points per player's possession", 'field goals made', 'field goals attempted', 'field goals, %', '3-pt field goals made', '3-pt field goals attempted', '3-pt field goals, %', 'free throws made', 'free throws attempted', 'free throws, %', 'rebounds', 'offensive rebounds', 'defensive rebounds', 'assists', 'steals', 'turnovers', 'blocks', 'fouls', 'fouls drawn', 'plus/minus', '2-pt field goals made', '2-pt field goals attempted', '2-pt field goals, %', 'points off assists', 'screen assist', 'points off screen assists', "number of player's possessions", 'team points with player', 'offensive rating', 'opponent possessions played', "opponent's points with player", 'defensive rating', 'net rating', 'assists to turnovers', 'steals to turnovers', 'draw foul rate', 'true shooting percentage', 'effective field goal percentage', 'uncontested field goals made', 'uncontested field goals', 'contested field goals made', 'contested field goals', "opponent's field goals made", "opponent's fie

We can observe from the histograms that many of these features are on very different scales. Hence, it might be beneficial to apply some standardization to the "train_data.csv" file before using it for training machine learning models.

The exported "train_data.csv" file now has the following properties:
* Many irrelevant features removed.
* Normalized by minutes played (if NORMALIZE = True)
* NOT standardized, needs to be done before training.
* Zero-inflated features have NOT yet been dealt with.

Start by investigating why effective field goals percentage so low for many players:

In [52]:
# First, filter out players where effective field goal percentage is zero or very small.
low_efg_df = df[df['effective field goal percentage'] < 0.3].copy()
# Focus on columns on which eFG depends on: 'field goals attempted', 'field goals made', and '3-pt made'
features = ['field goals attempted', 'field goals made', '3-pt field goals made', 'effective field goal percentage']
low_efg_df = low_efg_df[features]
print(low_efg_df.head())


    field goals attempted  field goals made  3-pt field goals made  \
22                    3.8              2.60                    0.0   
44                    8.0              4.00                    0.0   
48                    7.0              3.80                    0.0   
54                    2.7              1.78                    0.0   
69                    5.7              3.60                    0.0   

    effective field goal percentage  
22                              0.0  
44                              0.0  
48                              0.0  
54                              0.0  
69                              0.0  


Seems like the data is just sometimes missing. Fix in in the "preprocess_data.ipynb" notebook by manually caluclating the effective field goal percentage.